# 04. Рыночные данные и макро-контроли
**Выходы:**
- `imoex_hv30.csv` — IMOEX + HV_30 (зависимая переменная)
- `macro_controls.csv` — ставка ЦБ, Brent, USD/RUB, RVI (дневные)
- `stock_prices.csv` — дневные котировки 60 компаний

**Источники:** MOEX ISS API (через apimoex), ЦБ РФ XML API

**Зависимая переменная:**  
`HV_30 = rolling(30).std(log_return) * sqrt(252)`

In [ ]:
# ── ЯЧЕЙКА 1: Установка ──────────────────────────────────────
!pip install apimoex requests -q

In [ ]:
# ── ЯЧЕЙКА 2: Настройки ──────────────────────────────────────
import requests
import apimoex
import pandas as pd
import numpy as np
from datetime import date

DATE_FROM = '2022-06-01'
DATE_TO   = '2025-12-31'

# 60 тикеров выборки
TICKERS = [
    'LKOH', 'SIBN', 'GAZP', 'ROSN', 'NVTK', 'SNGS', 'SNGSP', 'TRNFP', 'BANE', 'RASP',
    'GMKN', 'NLMK', 'CHMF', 'MAGN', 'ALRS', 'PLZL', 'RUAL', 'ENPG', 'UGLD', 'MTLR',
    'MTLRP', 'SBER', 'SBERP', 'VTBR', 'CBOM', 'SVCB', 'BSPB', 'MOEX', 'RENI', 'DOMRF',
    'T', 'MTSS', 'RTKM', 'VKCO', 'YDEX', 'OZON', 'POSI', 'HEAD', 'CNRU',
    'X5', 'MGNT', 'FIXP', 'MDMG', 'GCHE', 'NKHP',
    'PIKK', 'LSRG', 'SMLT',
    'IRAO', 'MSNG', 'TGKA', 'DVEC',
    'AFLT', 'FLOT', 'FESH', 'AFKS', 'PHOR', 'TATN', 'TATNP', 'KAZT',
]

print(f'Тикеров: {len(TICKERS)}')
print(f'Период: {DATE_FROM} — {DATE_TO}')

In [ ]:
# ── ЯЧЕЙКА 3: IMOEX + HV_30 ──────────────────────────────────

with requests.Session() as session:
    imoex_data = apimoex.get_market_history(
        session, 'IMOEX',
        start=DATE_FROM, end=DATE_TO,
        market='index',
        columns=('TRADEDATE', 'CLOSE'),
    )

imoex = pd.DataFrame(imoex_data)
imoex.columns = ['date', 'close']
imoex['date']     = pd.to_datetime(imoex['date'])
imoex             = imoex.sort_values('date').reset_index(drop=True)
imoex['log_ret']  = np.log(imoex['close'] / imoex['close'].shift(1))
imoex['HV_30']    = imoex['log_ret'].rolling(30).std() * np.sqrt(252)

imoex.to_csv('/content/imoex_hv30.csv', index=False, encoding='utf-8-sig')
print(f'IMOEX: {len(imoex)} торговых дней')
print(imoex[['date', 'close', 'HV_30']].tail(5).to_string())

In [ ]:
# ── ЯЧЕЙКА 4: RVI (индекс волатильности MOEX) ────────────────

with requests.Session() as session:
    rvi_data = apimoex.get_market_history(
        session, 'RVI',
        start=DATE_FROM, end=DATE_TO,
        market='index',
        columns=('TRADEDATE', 'CLOSE'),
    )

rvi = pd.DataFrame(rvi_data)
rvi.columns = ['date', 'rvi']
rvi['date'] = pd.to_datetime(rvi['date'])
print(f'RVI: {len(rvi)} наблюдений')
print(rvi.tail(3).to_string())

In [ ]:
# ── ЯЧЕЙКА 5: USD/RUB — ЦБ РФ XML API ───────────────────────
from xml.etree import ElementTree as ET

def get_cbr_usd(date_from, date_to):
    url = (
        f'https://www.cbr.ru/scripts/XML_dynamic.asp'
        f'?date_req1={pd.to_datetime(date_from).strftime("%d/%m/%Y")}'
        f'&date_req2={pd.to_datetime(date_to).strftime("%d/%m/%Y")}'
        f'&VAL_NM_RQ=R01235'
    )
    r = requests.get(url, timeout=30)
    root = ET.fromstring(r.text)
    records = []
    for rec in root.findall('Record'):
        records.append({
            'date':    pd.to_datetime(rec.attrib['Date'], dayfirst=True),
            'usd_rub': float(rec.find('Value').text.replace(',', '.')),
        })
    return pd.DataFrame(records)

usd = get_cbr_usd(DATE_FROM, DATE_TO)
print(f'USD/RUB: {len(usd)} наблюдений')
print(usd.tail(3).to_string())

In [ ]:
# ── ЯЧЕЙКА 6: Ставка ЦБ РФ ───────────────────────────────────
# Источник: ЦБ РФ, история ключевой ставки
# https://www.cbr.ru/hd_base/KeyRate/
# Загружаем как step function (ставка действует до следующего изменения)

cbr_rate_url = 'https://www.cbr.ru/hd_base/KeyRate/'
tables = pd.read_html(cbr_rate_url)
rate_df = tables[0].copy()
rate_df.columns = ['date', 'rate']
rate_df['date'] = pd.to_datetime(rate_df['date'], dayfirst=True)
rate_df['rate'] = rate_df['rate'].astype(str).str.replace(',', '.').astype(float)
rate_df = rate_df.sort_values('date').reset_index(drop=True)

# Разворачиваем в дневной ряд (forward fill)
date_range = pd.date_range(DATE_FROM, DATE_TO, freq='D')
rate_daily = (
    rate_df.set_index('date')
    .reindex(date_range)
    .fillna(method='ffill')
    .rename_axis('date')
    .reset_index()
)
rate_daily.columns = ['date', 'cbr_rate']
print(f'Ключевая ставка: {len(rate_daily)} дней')
print(rate_daily.tail(3).to_string())

In [ ]:
# ── ЯЧЕЙКА 7: Brent ───────────────────────────────────────────
# TODO: выбрать источник
# Вариант A — через investing.com / stooq (pandas_datareader)
# Вариант B — через MOEX (фьючерс BR)
# Вариант C — загрузить CSV вручную с EIA или Macrotrends

# Пример для варианта B (MOEX фьючерс, может не покрывать весь период):
# with requests.Session() as session:
#     brent_data = apimoex.get_market_history(
#         session, 'BRH5',  # ← актуальный тикер фьючерса!
#         start=DATE_FROM, end=DATE_TO,
#         market='forts',
#         engine='futures',
#         columns=('TRADEDATE', 'CLOSE'),
#     )

# TODO: реализовать и раскомментировать
brent = pd.DataFrame({'date': pd.date_range(DATE_FROM, DATE_TO, freq='D'), 'brent': None})
print('⚠️  Brent — TODO')

In [ ]:
# ── ЯЧЕЙКА 8: Объединяем макро-контроли ──────────────────────

macro = (
    imoex[['date', 'close', 'log_ret', 'HV_30']]
    .merge(rvi,       on='date', how='left')
    .merge(usd,       on='date', how='left')
    .merge(rate_daily, on='date', how='left')
    .merge(brent[['date', 'brent']], on='date', how='left')
)

# Forward fill для нерабочих дней ставки и USD/RUB
macro[['usd_rub', 'cbr_rate']] = macro[['usd_rub', 'cbr_rate']].fillna(method='ffill')

macro.to_csv('/content/macro_controls.csv', index=False, encoding='utf-8-sig')
print(f'Макро датасет: {len(macro)} строк x {len(macro.columns)} колонок')
print(macro.isnull().sum().to_string())

In [ ]:
# ── ЯЧЕЙКА 9: Котировки 60 компаний ──────────────────────────

all_prices = []

with requests.Session() as session:
    for ticker in TICKERS:
        try:
            data = apimoex.get_security_history(
                session, ticker,
                start=DATE_FROM, end=DATE_TO,
                columns=('TRADEDATE', 'CLOSE', 'VOLUME'),
            )
            if data:
                df_t = pd.DataFrame(data)
                df_t.columns = ['date', 'close', 'volume']
                df_t['ticker'] = ticker
                all_prices.append(df_t)
        except Exception as e:
            print(f'  ✗ {ticker}: {e}')

prices = pd.concat(all_prices, ignore_index=True)
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)

prices.to_csv('/content/stock_prices.csv', index=False, encoding='utf-8-sig')
print(f'\n✓ Котировки: {len(prices)} строк, {prices["ticker"].nunique()} тикеров')
print(f'Пропуски: {prices[prices["close"].isna()]["ticker"].value_counts().head(10).to_string()}')